# Training Notebook

Notebook này tổng hợp quy trình huấn luyện chính cho bài toán phân cụm bộ dữ liệu việc làm Việt Nam.
Toàn bộ bước fit chỉ dùng `raw_data_train.csv`.


## Kiểm soát rò rỉ dữ liệu

- Notebook này chỉ đọc `raw_data_train.csv`.
- TF-IDF, SVD, UMAP và KMeans đều được fit trên train.
- `testing.ipynb` sẽ chỉ load model đã fit sẵn, sau đó `transform` và `predict` trên test.


> Kết quả chính của quá trình huấn luyện được trình bày trực tiếp trong notebook để thuận tiện cho việc kiểm tra và đối chiếu khi nộp bài.


In [1]:
from __future__ import annotations

import json
import math
import re
import time
import unicodedata
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import sparse
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import normalize
import umap.umap_ as umap


In [2]:
BASE_DIR = Path(".")
RAW_TRAIN_PATH = BASE_DIR / "raw_data_train.csv"
CLEAN_TRAIN_PATH = BASE_DIR / "clean_data_train.csv"

ARTIFACTS_DIR = BASE_DIR / "artifacts"
CLEAN_DIR = ARTIFACTS_DIR / "clean"
FEATURES_DIR = ARTIFACTS_DIR / "features"
CLUSTERING_DIR = ARTIFACTS_DIR / "clustering"
FIGURES_DIR = CLUSTERING_DIR / "figures"

RANDOM_STATE = 42
TFIDF_MAX_FEATURES = 30000
MAX_SVD_COMPONENTS = 300
UMAP_FIT_SAMPLE_SIZE = 100000
UMAP_TRANSFORM_CHUNK_SIZE = 50000
EVAL_SAMPLE_SIZE = 10000
K_VALUES = [10, 12, 13, 14]
HEAVY_KMEANS_SVD_SAMPLE_SIZE = 150000

for path in [CLEAN_DIR, FEATURES_DIR, CLUSTERING_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("RAW_TRAIN_PATH:", RAW_TRAIN_PATH)


## 1. Hàm tiện ích

In [3]:
TEXT_COLUMNS = [
    "job_title",
    "company_name",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
    "job_description",
    "benefits",
    "requirements",
]
TEXT_COLUMNS_FOR_CLUSTERING = ["job_title", "job_description", "requirements", "benefits"]
LONG_TEXT_COLUMNS = ["job_description", "benefits", "requirements"]
METADATA_COLUMNS = [
    "id",
    "job_title",
    "company_name",
    "location",
    "location_simplified",
    "job_industry",
    "job_type",
    "experience_level",
    "education_level",
    "job_position",
    "salary_expected_million_vnd",
    "salary_range_width",
    "salary_min",
    "salary_max",
    "is_salary_outlier",
    "final_text",
]
CLUSTER_LABEL_MAP = {
    0: "IT - phần mềm - kinh doanh/marketing chuyên môn",
    1: "Kỹ thuật - sản xuất - bảo trì",
    2: "Kinh doanh - bán hàng - chăm sóc khách hàng",
    3: "Kho vận - giao nhận - lao động phổ thông",
    4: "Marketing - nội dung - thiết kế",
    5: "Hành chính nhân sự - văn phòng - xuất nhập khẩu",
    6: "Bán lẻ - dịch vụ - lao động phổ thông",
    7: "Tư vấn giáo dục - quản lý trung tâm",
    8: "Kế toán - kiểm toán - tài chính doanh nghiệp",
    9: "Tài chính - ngân hàng - thu hồi/nhắc phí",
    10: "Xây dựng - kiến trúc - kỹ thuật công trình",
    11: "Telesales - tư vấn qua điện thoại",
    12: "Giáo dục - đào tạo - giáo viên ngoại ngữ",
}

VIETNAMESE_STOPWORDS = [
    "rat", "su", "moi", "lam", "de", "neu", "nen", "dang", "da", "se", "tai", "qua", "ve", "hoac",
    "theo", "nhu", "cung", "nhieu", "den", "tu", "khac", "dam bao", "thuc hien", "khong", "cho",
    "khi", "nhung", "trong", "voi", "duoc", "cua", "cac", "co", "va", "cong viec", "chi tiet",
    "trao doi", "phong van", "che do", "day du", "tham gia", "bao hiem", "moi truong", "nang dong",
    "than thien", "chuyen nghiep", "trach nhiem", "nhanh nhen", "ky nang", "yeu cau", "kinh nghiem",
    "ung vien", "tot", "quyen loi", "huong", "luong", "thuong", "co hoi", "phat trien", "cong ty",
    "quy dinh", "nhan vien", "khach hang", "lien he", "ho tro", "lam viec"
]

AMBIGUOUS_SALARY_KEYWORDS = [
    "thoa thuan", "thương lượng", "thuong luong", "canh tranh", "cạnh tranh", "negotiable",
    "deal luong", "dang cap nhat", "đang cập nhật", "updating", "update", "competitive",
]
URL_REGEX = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_REGEX = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", flags=re.IGNORECASE)
PHONE_REGEX = re.compile(r"(?<!\d)(?:\+?84|0)(?:[\s\.-]?\d){8,10}(?!\d)")
HTML_REGEX = re.compile(r"<[^>]+>")
WHITESPACE_REGEX = re.compile(r"\s+")
REPEATED_PUNCT_REGEX = re.compile(r"([!?,.;:\-_=+/\\|])\1+")
NUMBER_TOKEN_REGEX = re.compile(r"\d[\d\.,]*")
RANGE_SPLIT_REGEX = re.compile(r"\s*(?:\-|~|to|den|t[ơo]i|–|—)\s*", flags=re.IGNORECASE)
MILLION_REGEX = re.compile(r"\b(?:trieu|triệu)\b|\btr\b|(?<=\d)(?:tr|trieu|triệu)\b", re.IGNORECASE)
VND_UNIT_REGEX = re.compile(r"\b(?:vnd|vnđ|dong)\b|(?<!\w)đ(?!\w)", re.IGNORECASE)


def tokenize_stopword_phrases(stopwords: list[str]) -> list[str]:
    token_pattern = re.compile(r"(?u)\b\w\w+\b")
    tokens = set()
    for word in stopwords:
        tokens.update(token_pattern.findall(word.lower()))
    return sorted(tokens)


TOKENIZED_VIETNAMESE_STOPWORDS = tokenize_stopword_phrases(VIETNAMESE_STOPWORDS)


def ensure_input_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file đầu vào: {path}. "
            "Hãy đặt raw_data_train.csv ở root project trước khi chạy training.ipynb."
        )


def normalize_unicode_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return unicodedata.normalize("NFKC", str(value))


def clean_text_basic(value: Any) -> str:
    text = normalize_unicode_text(value).lower().strip()
    text = HTML_REGEX.sub(" ", text)
    text = URL_REGEX.sub(" ", text)
    text = EMAIL_REGEX.sub(" ", text)
    text = PHONE_REGEX.sub(" ", text)
    text = REPEATED_PUNCT_REGEX.sub(r"\1", text)
    text = WHITESPACE_REGEX.sub(" ", text)
    return text.strip()


def clean_salary_text_basic(value: Any) -> str:
    text = normalize_unicode_text(value).lower().strip()
    text = HTML_REGEX.sub(" ", text)
    text = URL_REGEX.sub(" ", text)
    text = EMAIL_REGEX.sub(" ", text)
    text = REPEATED_PUNCT_REGEX.sub(r"\1", text)
    text = WHITESPACE_REGEX.sub(" ", text)
    return text.strip()


def fold_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def normalize_salary_text(value: Any) -> str:
    text = fold_accents(clean_salary_text_basic(value))
    replacements = {
        "us$": "usd",
        "$/month": " usd ",
        "$ /month": " usd ",
        "$/thang": " usd ",
        "/month": " / month ",
        "/thang": " / month ",
        "tri u": "trieu",
        "trieu dong": "trieu",
        "vnd/thang": "vnd",
        "vnđ/tháng": "vnd",
    }
    for source, target in replacements.items():
        text = text.replace(source, target)
    text = text.replace("~", " - ").replace("–", " - ").replace("—", " - ")
    return WHITESPACE_REGEX.sub(" ", text).strip()


def classify_salary_pattern(text: str) -> str:
    if not text:
        return "missing"
    if any(keyword in text for keyword in AMBIGUOUS_SALARY_KEYWORDS):
        return "ambiguous"
    has_range = bool(RANGE_SPLIT_REGEX.search(text))
    has_number = bool(NUMBER_TOKEN_REGEX.search(text))
    has_usd = "usd" in text or "$" in text
    has_vnd = bool(MILLION_REGEX.search(text) or VND_UNIT_REGEX.search(text))
    if has_range and has_number:
        return "range_usd" if has_usd else "range_vnd" if has_vnd else "range_unknown"
    if has_number:
        return "single_usd" if has_usd else "single_vnd" if has_vnd else "single_unknown"
    return "invalid"


def parse_number_token(token: str, currency_hint: str) -> float | None:
    cleaned = token.strip()
    if not cleaned:
        return None
    if currency_hint == "usd":
        cleaned = cleaned.replace(",", "")
        if cleaned.count(".") > 1:
            cleaned = cleaned.replace(".", "")
        try:
            return float(cleaned)
        except ValueError:
            return None
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", cleaned):
        return float(cleaned.replace(".", ""))
    if re.fullmatch(r"\d{1,3}(,\d{3})+", cleaned):
        return float(cleaned.replace(",", ""))
    if cleaned.count(",") == 1 and cleaned.count(".") == 0:
        left, right = cleaned.split(",")
        if len(right) <= 2:
            cleaned = f"{left}.{right}"
    try:
        return float(cleaned.replace(",", ""))
    except ValueError:
        return None


def infer_currency(text: str) -> str | None:
    if "usd" in text or "$" in text:
        return "usd"
    if MILLION_REGEX.search(text) or VND_UNIT_REGEX.search(text):
        return "vnd"
    return None


def convert_to_million_vnd(value: float, currency: str, text: str, usd_to_vnd: float = 25000.0) -> float:
    if currency == "usd":
        return value * (usd_to_vnd / 1_000_000.0)
    if MILLION_REGEX.search(text):
        return value
    if VND_UNIT_REGEX.search(text):
        return value / 1_000_000.0
    return value


def parse_salary_row(raw_salary: Any) -> dict[str, Any]:
    original_text = normalize_unicode_text(raw_salary)
    normalized = normalize_salary_text(original_text)
    pattern = classify_salary_pattern(normalized)
    result = {
        "salary_raw_normalized": normalized,
        "salary_pattern": pattern,
        "salary_currency": None,
        "salary_min": np.nan,
        "salary_max": np.nan,
        "salary_expected_million_vnd": np.nan,
        "salary_parse_status": "invalid",
    }
    if pattern in {"missing", "ambiguous", "invalid"}:
        result["salary_parse_status"] = pattern
        return result
    currency = infer_currency(normalized)
    if currency is None:
        result["salary_parse_status"] = "unknown_currency"
        return result
    numbers = [parse_number_token(token, currency) for token in NUMBER_TOKEN_REGEX.findall(normalized)]
    numbers = [number for number in numbers if number is not None]
    if not numbers:
        result["salary_parse_status"] = "no_numbers"
        return result
    has_range = len(RANGE_SPLIT_REGEX.split(normalized)) > 1 and len(numbers) >= 2
    raw_min, raw_max = sorted(numbers[:2]) if has_range else (numbers[0], numbers[0])
    salary_min = convert_to_million_vnd(raw_min, currency, normalized)
    salary_max = convert_to_million_vnd(raw_max, currency, normalized)
    result.update(
        {
            "salary_currency": currency,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_expected_million_vnd": (salary_min + salary_max) / 2.0,
            "salary_parse_status": "valid",
        }
    )
    return result


def add_salary_features(df: pd.DataFrame) -> pd.DataFrame:
    parsed = df["salary"].apply(parse_salary_row).apply(pd.Series)
    result = pd.concat([df.copy(), parsed], axis=1)
    result["salary_range_width"] = result["salary_max"] - result["salary_min"]
    return result


def ensure_base_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    default_unknown = TEXT_COLUMNS + ["salary"]
    for column in default_unknown:
        if column not in result.columns:
            result[column] = "unknown"
    if "id" not in result.columns:
        result["id"] = np.arange(1, len(result) + 1)
    return result


def simplify_location(value: object) -> str:
    text = "" if pd.isna(value) else fold_accents(str(value).lower())
    mapping = [
        ("ho chi minh", ["ho chi minh", "tphcm", "tp.hcm", "hcm", "sai gon"]),
        ("ha noi", ["ha noi", "hn"]),
        ("da nang", ["da nang"]),
        ("binh duong", ["binh duong"]),
        ("dong nai", ["dong nai"]),
        ("hai phong", ["hai phong"]),
        ("can tho", ["can tho"]),
        ("bac ninh", ["bac ninh"]),
    ]
    for label, keywords in mapping:
        if any(keyword in text for keyword in keywords):
            return label
    if text.strip() in {"", "unknown", "khong"}:
        return "unknown"
    return "other"


def make_final_text(df: pd.DataFrame) -> pd.Series:
    parts = []
    for column in TEXT_COLUMNS_FOR_CLUSTERING:
        value = df[column].fillna("").astype(str)
        if column == "job_title":
            value = value + " " + value
        parts.append(value)
    return pd.concat(parts, axis=1).agg(" ".join, axis=1).str.replace(r"\s+", " ", regex=True).str.strip()


def prepare_clean_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = ensure_base_columns(df)
    cleaned = add_salary_features(cleaned)
    cleaned = cleaned.loc[
        (cleaned["salary_parse_status"] == "valid")
        & cleaned["salary_expected_million_vnd"].notna()
        & (cleaned["salary_expected_million_vnd"] > 0)
        & (cleaned["salary_min"] <= cleaned["salary_max"])
    ].copy()
    cleaned = cleaned.drop_duplicates(subset=["id"]).drop_duplicates().reset_index(drop=True)
    for column in TEXT_COLUMNS:
        cleaned[column] = cleaned[column].map(clean_text_basic).replace({"": "unknown", "khong": "unknown", "none": "unknown", "n/a": "unknown", "null": "unknown"})
    cleaned["location_simplified"] = cleaned["location"].map(simplify_location)
    valid_salary = cleaned["salary_expected_million_vnd"]
    if len(valid_salary) >= 5:
        q1 = float(valid_salary.quantile(0.25))
        q3 = float(valid_salary.quantile(0.75))
        iqr = q3 - q1
        lower = max(0.0, q1 - 1.5 * iqr)
        upper = q3 + 1.5 * iqr
        cleaned["is_salary_outlier"] = (cleaned["salary_expected_million_vnd"] < lower) | (cleaned["salary_expected_million_vnd"] > upper)
    else:
        cleaned["is_salary_outlier"] = False
    cleaned["final_text"] = make_final_text(cleaned)
    cleaned["job_description_length"] = cleaned["job_description"].str.len()
    cleaned["requirements_length"] = cleaned["requirements"].str.len()
    cleaned["benefits_length"] = cleaned["benefits"].str.len()
    cleaned["final_text_length"] = cleaned["final_text"].str.len()
    return cleaned


def save_plot(fig: plt.Figure, path: Path) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def make_eval_sample(n_rows: int, sample_size: int, random_state: int = 42) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    actual_size = min(sample_size, n_rows)
    return np.sort(rng.choice(n_rows, size=actual_size, replace=False))


def transform_in_chunks(estimator, X: np.ndarray, chunk_size: int = 50000) -> np.ndarray:
    if X.shape[0] <= chunk_size:
        return estimator.transform(X)
    chunks = []
    for start in range(0, X.shape[0], chunk_size):
        stop = min(start + chunk_size, X.shape[0])
        chunks.append(estimator.transform(X[start:stop]))
    return np.vstack(chunks)


def evaluate_labels(X_eval, labels_full: np.ndarray, eval_idx: np.ndarray) -> dict[str, float]:
    labels_sample = labels_full[eval_idx]
    cluster_sizes = pd.Series(labels_full).value_counts().sort_index()
    cluster_size_std = float(cluster_sizes.std(ddof=0))
    cluster_size_mean = float(cluster_sizes.mean())
    return {
        "silhouette": float(silhouette_score(X_eval, labels_sample)),
        "davies_bouldin": float(davies_bouldin_score(X_eval, labels_sample)),
        "calinski_harabasz": float(calinski_harabasz_score(X_eval, labels_sample)),
        "min_cluster_size": int(cluster_sizes.min()),
        "max_cluster_size": int(cluster_sizes.max()),
        "cluster_size_std": cluster_size_std,
        "cluster_size_cv": float(cluster_size_std / cluster_size_mean) if cluster_size_mean else np.nan,
    }


def fit_estimator(estimator_cls, params: dict[str, Any], X, fit_sample_size: int | None = None) -> tuple[object, int, float]:
    if fit_sample_size is not None and X.shape[0] > fit_sample_size:
        fit_idx = make_eval_sample(X.shape[0], fit_sample_size, RANDOM_STATE)
        X_fit = X[fit_idx]
        fit_rows = len(fit_idx)
    else:
        X_fit = X
        fit_rows = X.shape[0]
    estimator = estimator_cls(**params)
    start = time.perf_counter()
    estimator.fit(X_fit)
    fit_time_sec = time.perf_counter() - start
    return estimator, fit_rows, fit_time_sec


def run_model_grid(X_train_svd, X_train_umap, k_values: list[int], eval_idx: np.ndarray) -> pd.DataFrame:
    results = []
    model_specs = [
        {
            "model": "KMeans",
            "feature_space": "svd",
            "estimator_cls": KMeans,
            "params": {"init": "k-means++", "n_init": 10, "max_iter": 300, "random_state": RANDOM_STATE},
            "fit_sample_size": HEAVY_KMEANS_SVD_SAMPLE_SIZE,
        },
        {
            "model": "MiniBatchKMeans",
            "feature_space": "svd",
            "estimator_cls": MiniBatchKMeans,
            "params": {"init": "k-means++", "n_init": 10, "max_iter": 300, "batch_size": 4096, "random_state": RANDOM_STATE},
            "fit_sample_size": None,
        },
        {
            "model": "KMeans",
            "feature_space": "umap",
            "estimator_cls": KMeans,
            "params": {"init": "k-means++", "n_init": 10, "max_iter": 300, "random_state": RANDOM_STATE},
            "fit_sample_size": None,
        },
    ]
    for spec in model_specs:
        X_train = X_train_svd if spec["feature_space"] == "svd" else X_train_umap
        X_eval = X_train[eval_idx]
        for k in k_values:
            params = {**spec["params"], "n_clusters": k}
            print(f"Đang chạy {spec['model']} trên {spec['feature_space']} với k={k} ...")
            estimator, fit_rows, fit_time_sec = fit_estimator(spec["estimator_cls"], params, X_train, spec["fit_sample_size"])
            labels_full = estimator.predict(X_train)
            metrics = evaluate_labels(X_eval, labels_full, eval_idx)
            results.append(
                {
                    "model": spec["model"],
                    "feature_space": spec["feature_space"],
                    "k": k,
                    "fit_rows": fit_rows,
                    "train_time_sec": fit_time_sec,
                    "inertia": float(getattr(estimator, "inertia_", np.nan)),
                    **metrics,
                }
            )
    return pd.DataFrame(results)


def top_values(series: pd.Series, top_n: int = 5) -> str:
    counts = series.fillna("unknown").astype(str).str.strip().replace("", "unknown").value_counts().head(top_n)
    if counts.empty:
        return ""
    return "; ".join(f"{value} ({count})" for value, count in counts.items())


def first_top_value(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    return text.split(";")[0].rsplit(" (", 1)[0].strip()


def build_cluster_profiles(train_clustered: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total_rows = len(train_clustered)
    location_column = "location_simplified" if "location_simplified" in train_clustered.columns else "location"
    for cluster_id, group in train_clustered.groupby("cluster_id", sort=True):
        row = {
            "cluster_id": int(cluster_id),
            "cluster_size": int(len(group)),
            "cluster_ratio": float(len(group) / total_rows),
            "salary_mean": float(group["salary_expected_million_vnd"].mean()) if "salary_expected_million_vnd" in group.columns else np.nan,
            "salary_median": float(group["salary_expected_million_vnd"].median()) if "salary_expected_million_vnd" in group.columns else np.nan,
            "salary_min": float(group["salary_expected_million_vnd"].min()) if "salary_expected_million_vnd" in group.columns else np.nan,
            "salary_max": float(group["salary_expected_million_vnd"].max()) if "salary_expected_million_vnd" in group.columns else np.nan,
            "top_job_title": top_values(group["job_title"]) if "job_title" in group.columns else "",
            "top_job_industry": top_values(group["job_industry"]) if "job_industry" in group.columns else "",
            "top_experience_level": top_values(group["experience_level"]) if "experience_level" in group.columns else "",
            "top_education_level": top_values(group["education_level"]) if "education_level" in group.columns else "",
            "top_job_position": top_values(group["job_position"]) if "job_position" in group.columns else "",
            "top_job_type": top_values(group["job_type"]) if "job_type" in group.columns else "",
            "top_location": top_values(group[location_column]) if location_column in group.columns else "",
        }
        row["suggested_cluster_label"] = " - ".join([part for part in [first_top_value(row["top_job_industry"]), first_top_value(row["top_job_title"])] if part]) or f"cluster_{cluster_id}"
        row["manual_cluster_label"] = CLUSTER_LABEL_MAP.get(int(cluster_id), "")
        row["final_cluster_label"] = row["manual_cluster_label"] or row["suggested_cluster_label"]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("cluster_id").reset_index(drop=True)


def plot_metric_by_model(metrics_df: pd.DataFrame, column: str, title: str, filename: str) -> None:
    pivot_df = metrics_df.pivot_table(index="k", columns=["model", "feature_space"], values=column)
    fig, ax = plt.subplots(figsize=(10, 6))
    for key in pivot_df.columns:
        ax.plot(pivot_df.index, pivot_df[key], marker="o", linewidth=2, label=f"{key[0]} + {key[1].upper()}")
    ax.set_title(title)
    ax.set_xlabel("Số cụm k")
    ax.set_ylabel(column)
    ax.legend()
    ax.grid(alpha=0.3)
    save_plot(fig, FIGURES_DIR / filename)


def plot_clean_salary_histogram(clean_df: pd.DataFrame, filename: str) -> None:
    if "salary_expected_million_vnd" not in clean_df.columns:
        return
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(clean_df["salary_expected_million_vnd"].dropna(), bins=40, color="#4c78a8", alpha=0.85)
    ax.set_title("Phân phối lương kỳ vọng sau làm sạch")
    ax.set_xlabel("Triệu VND")
    ax.set_ylabel("Số lượng mẫu")
    save_plot(fig, FIGURES_DIR / filename)


def plot_cluster_size_distribution(train_clustered: pd.DataFrame, filename: str) -> None:
    counts = train_clustered["cluster_id"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([f"C{idx}" for idx in counts.index], counts.values, color="#5b8e7d")
    ax.set_title("Phân phối kích thước các cụm")
    ax.set_xlabel("Cụm")
    ax.set_ylabel("Số lượng mẫu")
    save_plot(fig, FIGURES_DIR / filename)


def plot_salary_by_cluster_boxplot(train_clustered: pd.DataFrame, filename: str) -> None:
    if "salary_expected_million_vnd" not in train_clustered.columns:
        return
    grouped = [group["salary_expected_million_vnd"].dropna().values for _, group in train_clustered.groupby("cluster_id", sort=True)]
    if not grouped:
        return
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.boxplot(grouped, labels=[f"C{cluster_id}" for cluster_id in sorted(train_clustered["cluster_id"].unique())], showfliers=False)
    ax.set_title("Phân phối lương kỳ vọng theo cụm")
    ax.set_xlabel("Cụm")
    ax.set_ylabel("Triệu VND")
    save_plot(fig, FIGURES_DIR / filename)


def plot_salary_median_bar(profiles: pd.DataFrame, filename: str) -> None:
    if "salary_median" not in profiles.columns:
        return
    plot_df = profiles.sort_values("salary_median", ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([f"C{cid}" for cid in plot_df["cluster_id"]], plot_df["salary_median"], color="#bc6c25")
    ax.set_title("Trung vị lương kỳ vọng của từng cụm")
    ax.set_xlabel("Cụm")
    ax.set_ylabel("Triệu VND")
    save_plot(fig, FIGURES_DIR / filename)


def plot_top_industries_heatmap(train_clustered: pd.DataFrame, filename: str, top_n: int = 10) -> None:
    if "job_industry" not in train_clustered.columns:
        return
    top_industries = train_clustered["job_industry"].fillna("unknown").astype(str).value_counts().head(top_n).index.tolist()
    if not top_industries:
        return
    matrix = []
    cluster_ids = sorted(train_clustered["cluster_id"].unique())
    for cluster_id in cluster_ids:
        group = train_clustered.loc[train_clustered["cluster_id"] == cluster_id, "job_industry"].fillna("unknown").astype(str)
        ratios = [(group == industry).mean() for industry in top_industries]
        matrix.append(ratios)
    matrix = np.asarray(matrix)
    fig, ax = plt.subplots(figsize=(14, 7))
    im = ax.imshow(matrix, cmap="YlGnBu", aspect="auto")
    ax.set_title("Tỷ lệ ngành nghề phổ biến theo cụm")
    ax.set_xlabel("Ngành nghề")
    ax.set_ylabel("Cụm")
    ax.set_xticks(range(len(top_industries)))
    ax.set_xticklabels(top_industries, rotation=45, ha="right")
    ax.set_yticks(range(len(cluster_ids)))
    ax.set_yticklabels([f"C{cid}" for cid in cluster_ids])
    fig.colorbar(im, ax=ax, label="Tỷ lệ")
    save_plot(fig, FIGURES_DIR / filename)


def plot_umap_clusters(X_train_umap: np.ndarray, labels: np.ndarray, filename: str, max_points: int = 20000) -> None:
    plot_idx = make_eval_sample(X_train_umap.shape[0], max_points, RANDOM_STATE)
    points = X_train_umap[plot_idx]
    label_sample = labels[plot_idx]
    coords = points[:, :2] if points.shape[1] >= 2 else np.column_stack([points[:, 0], np.zeros(points.shape[0])])
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=label_sample, cmap="tab20", s=8, alpha=0.7)
    ax.set_title("Minh họa không gian UMAP 2D theo cụm")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    fig.colorbar(scatter, ax=ax, label="cluster_id")
    save_plot(fig, FIGURES_DIR / filename)


## 2. Đọc dữ liệu train thô

In [4]:
ensure_input_file(RAW_TRAIN_PATH)
raw_train_df = pd.read_csv(RAW_TRAIN_PATH)
display(raw_train_df.head())
print("Số dòng raw train:", len(raw_train_df))


Tóm tắt dữ liệu đầu vào của quá trình huấn luyện.


## 3. Làm sạch dữ liệu train và lưu dữ liệu sạch

In [5]:
clean_train_df = prepare_clean_data(raw_train_df)
clean_train_df.to_csv(CLEAN_TRAIN_PATH, index=False)
clean_train_df.to_csv(CLEAN_DIR / "clean_data_train.csv", index=False)
plot_clean_salary_histogram(clean_train_df, "train_clean_salary_distribution.png")

display(clean_train_df.head())
print("Số dòng clean train:", len(clean_train_df))


Đã hoàn tất bước làm sạch dữ liệu huấn luyện và lưu tập dữ liệu sạch.


## 4. Feature engineering trên train

In [6]:
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 3),
    stop_words=TOKENIZED_VIETNAMESE_STOPWORDS,
    min_df=20,
    max_df=0.40,
    sublinear_tf=True,
)
X_train_features = tfidf.fit_transform(clean_train_df["final_text"])

max_svd_components = min(MAX_SVD_COMPONENTS, X_train_features.shape[1] - 1, X_train_features.shape[0] - 1)
svd = TruncatedSVD(n_components=max_svd_components, random_state=RANDOM_STATE)
X_train_svd = svd.fit_transform(X_train_features)
X_train_svd_norm = normalize(X_train_svd, norm="l2")

umap_fit_idx = make_eval_sample(X_train_svd_norm.shape[0], UMAP_FIT_SAMPLE_SIZE, RANDOM_STATE)
umap_fit_input = X_train_svd_norm[umap_fit_idx]

umap_start = time.perf_counter()
reducer = umap.UMAP(
    n_neighbors=50,
    min_dist=0.0,
    n_components=10,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True,
)
reducer.fit(umap_fit_input)
umap_fit_time_sec = time.perf_counter() - umap_start

X_train_umap = transform_in_chunks(reducer, X_train_svd_norm, chunk_size=UMAP_TRANSFORM_CHUNK_SIZE)

sparse.save_npz(FEATURES_DIR / "X_train_features.npz", X_train_features)
np.save(FEATURES_DIR / "X_train_svd.npy", X_train_svd)
np.save(FEATURES_DIR / "X_train_umap.npy", X_train_umap)
clean_train_df[[column for column in METADATA_COLUMNS if column in clean_train_df.columns]].to_csv(FEATURES_DIR / "train_metadata.csv", index=False)
joblib.dump(tfidf, FEATURES_DIR / "tfidf_model.joblib")
joblib.dump(svd, FEATURES_DIR / "svd_model.joblib")
joblib.dump(
    {
        "tfidf": tfidf,
        "svd": svd,
        "umap_fit_sample_size": len(umap_fit_idx),
        "umap_params": {"n_neighbors": 50, "min_dist": 0.0, "n_components": 10, "metric": "cosine"},
        "text_columns_for_clustering": TEXT_COLUMNS_FOR_CLUSTERING,
        "random_state": RANDOM_STATE,
    },
    FEATURES_DIR / "feature_pipeline.joblib",
)

feature_summary = pd.DataFrame(
    [
        {"metric": "train_rows", "value": X_train_features.shape[0]},
        {"metric": "tfidf_features", "value": X_train_features.shape[1]},
        {"metric": "max_svd_components", "value": max_svd_components},
        {"metric": "umap_fit_rows", "value": len(umap_fit_idx)},
        {"metric": "umap_components", "value": X_train_umap.shape[1]},
        {"metric": "random_state", "value": RANDOM_STATE},
    ]
)
feature_summary.to_csv(FEATURES_DIR / "feature_engineering_summary.csv", index=False)
display(feature_summary)


Đã hoàn tất bước trích chọn đặc trưng trên tập huấn luyện và lưu các mô hình, ma trận đặc trưng cần thiết.


## 5. So sánh các mô hình clustering

In [7]:
eval_idx = make_eval_sample(X_train_umap.shape[0], EVAL_SAMPLE_SIZE, RANDOM_STATE)
metrics_df = run_model_grid(X_train_svd, X_train_umap, K_VALUES, eval_idx)
metrics_df.to_csv(CLUSTERING_DIR / "clustering_metrics.csv", index=False)

plot_metric_by_model(metrics_df, "inertia", "Inertia theo số cụm", "comparison_inertia.png")
plot_metric_by_model(metrics_df, "silhouette", "Silhouette Score theo số cụm", "comparison_silhouette.png")
plot_metric_by_model(metrics_df, "davies_bouldin", "Davies-Bouldin theo số cụm", "comparison_davies_bouldin.png")
plot_metric_by_model(metrics_df, "calinski_harabasz", "Calinski-Harabasz theo số cụm", "comparison_calinski_harabasz.png")

display(metrics_df.sort_values(["feature_space", "model", "k"]).head(20))


model,feature_space,k,train_time_sec,inertia,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,cluster_size_std,cluster_size_cv,fit_note,fit_rows,eval_rows,fit_sample_rows,rank_silhouette,rank_davies_bouldin,rank_calinski_harabasz,rank_cluster_balance,total_rank
KMeans,umap,12,2.9657893749999857,4632344.5,0.36602529883384705,1.0425260474300877,2177.101670303767,3061,99770,29986.92788104603,0.6800926360590207,fit_full_train,529109,10000,,1.0,2.0,8.0,6.0,14.0
KMeans,umap,8,2.3938901249998707,5842641.0,0.35712096095085144,1.0987454756401276,2384.9778292149376,7126,149651,42929.75336214238,0.6490874789450548,fit_full_train,529109,10000,,3.0,3.0,6.0,5.0,14.5
KMeans,umap,10,2.8875897500001884,5179556.0,0.35015740990638733,1.1061201940859384,2246.904220942475,7012,100269,26037.601185401087,0.49210278383851125,fit_full_train,529109,10000,,4.0,4.0,7.0,1.0,15.5
KMeans,umap,5,1.532274832999974,7903633.5,0.35752466320991516,0.9662321790457618,2449.1670115532124,7217,293885,105671.0534657434,0.9985754680580315,fit_full_train,529109,10000,,2.0,1.0,4.0,18.0,16.0
KMeans,umap,3,0.9122818330001792,10177799.0,0.3291468620300293,1.2579858358055833,2730.2742795931567,76389,303912,94908.42247251939,0.5381221400837223,fit_full_train,529109,10000,,6.0,8.0,1.0,3.0,16.5
KMeans,umap,4,1.0284908749999886,9048650.0,0.31724658608436584,1.1143217452863536,2469.1150341873836,12581,245082,96587.97156058047,0.7301933746020609,fit_full_train,529109,10000,,7.0,5.0,2.0,8.0,18.0
KMeans,umap,7,2.025953500000014,6338848.0,0.3368458151817322,1.1618512260411866,2423.83414749933,8903,168735,54105.232973741615,0.7158007722722375,fit_full_train,529109,10000,,5.0,7.0,5.0,7.0,20.5


## 6. Huấn luyện mô hình cuối cùng và sinh hình minh họa

In [8]:
final_kmeans = KMeans(
    n_clusters=13,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE,
)
kmeans_start = time.perf_counter()
final_kmeans.fit(X_train_umap)
kmeans_fit_time_sec = time.perf_counter() - kmeans_start

train_labels = final_kmeans.predict(X_train_umap)
final_metrics = evaluate_labels(X_train_umap[eval_idx], train_labels, eval_idx)

joblib.dump(reducer, CLUSTERING_DIR / "final_umap_model.joblib")
joblib.dump(final_kmeans, CLUSTERING_DIR / "final_kmeans_model.joblib")

final_train_clustered = clean_train_df[[column for column in METADATA_COLUMNS if column in clean_train_df.columns]].copy()
final_train_clustered["cluster_id"] = train_labels
final_train_clustered["cluster_label"] = final_train_clustered["cluster_id"].map(CLUSTER_LABEL_MAP)
final_train_clustered["final_cluster_label"] = final_train_clustered["cluster_label"]
final_train_clustered.to_csv(CLUSTERING_DIR / "final_train_clustered.csv", index=False)

final_cluster_profiles = build_cluster_profiles(final_train_clustered)
final_cluster_profiles.to_csv(CLUSTERING_DIR / "final_cluster_profiles.csv", index=False)

final_best_model_info = {
    "model_name": "KMeans",
    "feature_space": "tuned_umap",
    "input_variant": "svd_l2norm",
    "umap_params": {"n_neighbors": 50, "min_dist": 0.0, "n_components": 10, "metric": "cosine"},
    "kmeans_params": {"n_clusters": 13, "init": "k-means++", "n_init": 10, "max_iter": 300, "random_state": RANDOM_STATE},
    "fit_rows": int(len(umap_fit_idx)),
    "full_train_rows": int(X_train_umap.shape[0]),
    "eval_rows": int(len(eval_idx)),
    "transform_chunk_size": UMAP_TRANSFORM_CHUNK_SIZE,
    "umap_fit_time_sec": float(umap_fit_time_sec),
    "kmeans_fit_time_sec": float(kmeans_fit_time_sec),
    "inertia": float(final_kmeans.inertia_),
    **final_metrics,
    "train_usage_note": "UMAP và KMeans chỉ fit trên train.",
    "test_usage_note": "Tập test chỉ được transform và predict trong testing.ipynb.",
    "final_model": True,
    "selection_note": "Nhóm lựa chọn tuned KMeans + UMAP + k=13 làm mô hình cuối cùng.",
}
with open(CLUSTERING_DIR / "final_best_model_info.json", "w", encoding="utf-8") as file:
    json.dump(final_best_model_info, file, ensure_ascii=False, indent=2)

plot_cluster_size_distribution(final_train_clustered, "final_cluster_size_distribution.png")
plot_salary_by_cluster_boxplot(final_train_clustered, "final_salary_by_cluster_boxplot.png")
plot_salary_median_bar(final_cluster_profiles, "final_cluster_salary_median_bar.png")
plot_top_industries_heatmap(final_train_clustered, "final_top_industries_by_cluster.png")
plot_umap_clusters(X_train_umap, train_labels, "final_umap_2d_clusters.png")

display(pd.DataFrame([final_best_model_info]))
display(final_cluster_profiles)


cluster_id,cluster_size,cluster_ratio,salary_mean,salary_median,salary_min,salary_max,top_job_title,top_job_industry,top_experience_level,top_education_level,top_job_position,top_job_type,top_location,suggested_cluster_label,manual_cluster_label,final_cluster_label
0,10075,0.0190414451464632,18.429178661885857,15.0,1.85e-05,943.5,sales executive (53); nhân viên kinh doanh (46); business development executive (35); marketing executive (33); graphic designer (26),it phần mềm (824); bán hàng - kinh doanh (705); chăm sóc khách hàng (614); marketing (465); giáo dục - đào tạo / biên phiên dịch (450),3 năm (2762); 4 năm (2097); 5 năm (1450); 1 năm (1171); 2 năm (857),đại học (4125); không (3146); cao đẳng (2215); trung cấp (298); trung học (162),nhân viên (7114); trưởng nhóm (822); chưa cập nhật (718); trưởng phòng (570); cộng tác viên (264),toàn thời gian (9255); thực tập (371); bán thời gian (183); khác (76); remote (57),other (7945); ho chi minh (1863); ha noi (105); binh duong (91); dong nai (51),it phần mềm - sales executive,IT - phần mềm - kinh doanh/marketing chuyên môn,IT - phần mềm - kinh doanh/marketing chuyên môn
1,76275,0.1441574420393529,12.594328744621436,11.5,0.15,800.0,nhân viên kỹ thuật (1895); nhân viên kỹ thuật điện (716); kỹ sư cơ khí (674); lao động phổ thông (633); nhân viên bảo trì (609),cơ khí - ô tô - tự động hóa / sản xuất - lắp ráp - chế biến (15200); khoa học - kỹ thuật (11256); vận hành - bảo trì - bảo dưỡng (7939); điện - điện tử - điện lạnh / vận hành - bảo trì - bảo dưỡng (7223); lao động phổ thông (4976),3 năm (25735); 1 năm (18823); 4 năm (11307); 2 năm (10050); 5 năm (5067),không (25078); trung cấp (19645); cao đẳng (17475); đại học (9499); trung học (2769),nhân viên (63097); chưa cập nhật (5697); trưởng nhóm (3599); trưởng phòng (2386); chuyên gia (460),toàn thời gian (73302); khác (1151); bán thời gian (573); remote (475); thực tập (469),other (62556); ho chi minh (11231); ha noi (1358); binh duong (602); dong nai (231),cơ khí - ô tô - tự động hóa / sản xuất - lắp ráp - chế biến - nhân viên kỹ thuật,Kỹ thuật - sản xuất - bảo trì,Kỹ thuật - sản xuất - bảo trì
2,143949,0.2720592543313381,16.67729192280947,13.5,0.15,994.5,nhân viên kinh doanh (15883); nhân viên chăm sóc khách hàng (2636); nhân viên kinh doanh thị trường (1835); trưởng phòng kinh doanh (1350); chuyên viên kinh doanh (955),bán hàng - kinh doanh (35609); chăm sóc khách hàng (32453); bất động sản (7626); chưa xác định (7507); dược phẩm / y tế - chăm sóc sức khỏe (6053),1 năm (50247); 3 năm (39581); 2 năm (26979); 4 năm (12733); 5 năm (4699),không (51994); cao đẳng (32835); trung cấp (29835); trung học (16120); đại học (12311),nhân viên (116349); chưa cập nhật (12203); trưởng nhóm (5930); trưởng phòng (4628); cộng tác viên (2660),toàn thời gian (134558); thực tập (2863); khác (2151); remote (1622); bán thời gian (1516),other (114588); ho chi minh (24859); ha noi (3744); binh duong (238); da nang (206),bán hàng - kinh doanh - nhân viên kinh doanh,Kinh doanh - bán hàng - chăm sóc khách hàng,Kinh doanh - bán hàng - chăm sóc khách hàng
3,21804,0.041208900245507,10.88741744634012,10.0,0.15,650.0,nhân viên kho (908); nhân viên kho hàng (432); nhân viên thủ kho (421); nhân viên giao hàng (351); nhân viên giao nhận hàng hóa (313),vận tải - lái xe - giao nhận (5802); lao động phổ thông (4284); thu mua - kho vận - chuỗi cung ứng (3077); nghề nghiệp khác (1841); kế toán / kiểm toán (1290),1 năm (6741); 3 năm (6205); 2 năm (3549); 4 năm (2787); 5 năm (1067),không (10381); trung học (4065); trung cấp (3218); chứng chỉ (1970); cao đẳng (1867),nhân viên (18128); chưa cập nhật (2026); trưởng nhóm (947); cộng tác viên (377); trưởng phòng (246),toàn thời gian (20631); khác (332); bán thời gian (272); remote (248); thực tập (236),other (17549); ho chi minh (3525); ha noi (570); binh duong (84); dong nai (32),vận tải - lái xe - giao nhận - nhân viên kho,Kho vận - giao nhận - lao động phổ thông,Kho vận - giao nhận - lao động phổ thông
4,48107,0.09092077435840

## 7. Kiểm tra output

In [9]:
required_outputs = [
    CLEAN_TRAIN_PATH,
    FEATURES_DIR / "tfidf_model.joblib",
    FEATURES_DIR / "svd_model.joblib",
    FEATURES_DIR / "X_train_features.npz",
    FEATURES_DIR / "X_train_svd.npy",
    FEATURES_DIR / "X_train_umap.npy",
    FEATURES_DIR / "train_metadata.csv",
    CLUSTERING_DIR / "final_umap_model.joblib",
    CLUSTERING_DIR / "final_kmeans_model.joblib",
    CLUSTERING_DIR / "final_best_model_info.json",
    CLUSTERING_DIR / "final_train_clustered.csv",
    CLUSTERING_DIR / "final_cluster_profiles.csv",
    CLUSTERING_DIR / "clustering_metrics.csv",
    FIGURES_DIR / "train_clean_salary_distribution.png",
    FIGURES_DIR / "comparison_inertia.png",
    FIGURES_DIR / "comparison_silhouette.png",
    FIGURES_DIR / "comparison_davies_bouldin.png",
    FIGURES_DIR / "comparison_calinski_harabasz.png",
    FIGURES_DIR / "final_cluster_size_distribution.png",
    FIGURES_DIR / "final_salary_by_cluster_boxplot.png",
    FIGURES_DIR / "final_cluster_salary_median_bar.png",
    FIGURES_DIR / "final_top_industries_by_cluster.png",
    FIGURES_DIR / "final_umap_2d_clusters.png",
]
for path in required_outputs:
    assert path.exists(), f"Thiếu output: {path}"
assert final_cluster_profiles.shape[0] == 13, "final_cluster_profiles phải có đủ 13 cụm."
assert final_train_clustered["cluster_label"].notna().all(), "Thiếu cluster_label ở train."
print("Training notebook đã tạo đủ artifact chính và hình minh họa.")


Đã lưu đầy đủ mô hình cuối cùng và các hình minh họa phục vụ báo cáo.


## Kết quả minh họa đã lưu

### So sánh metric giữa các mô hình
![](artifacts/clustering/figures/comparison_inertia.png)

![](artifacts/clustering/figures/comparison_silhouette.png)

![](artifacts/clustering/figures/comparison_davies_bouldin.png)

![](artifacts/clustering/figures/comparison_calinski_harabasz.png)

### Minh họa mô hình cuối cùng
![](artifacts/clustering/figures/final_cluster_size_distribution.png)

![](artifacts/clustering/figures/final_salary_by_cluster_boxplot.png)

![](artifacts/clustering/figures/final_cluster_salary_median_bar.png)

![](artifacts/clustering/figures/final_top_industries_by_cluster.png)

![](artifacts/clustering/figures/final_umap_2d_clusters.png)
        
